In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# Read data
# Hlavný priečinok s datasetmi
ROOT = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches"
)

# Kódy líg
league_names = {
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "SP1": "La Liga",
    "F1": "Ligue 1",
}

dataframes = []
errors = []

for folder in sorted(ROOT.glob("big5_*")):
    if not folder.is_dir():
        continue

    season_code = folder.name.replace("big5_", "")

    # big5_1011 -> 2010/11
    start_year = 2000 + int(season_code[:2])
    end_year = 2000 + int(season_code[2:])
    season = f"{start_year}/{str(end_year)[-2:]}"

    for file in sorted(folder.glob("*.csv")):
        league_code = file.stem

        if league_code not in league_names:
            continue

        try:
            # Niektoré historické CSV môžu používať
            # odlišné kódovanie textu.
            try:
                df = pd.read_csv(file, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding="cp1252")

            df.columns = df.columns.str.strip()

            df["Season"] = season
            df["League"] = league_names[league_code]
            df["LeagueCode"] = league_code

            dataframes.append(df)

        except Exception as e:
            errors.append({
                "file": str(file),
                "error": str(e)
            })

if not dataframes:
    raise ValueError(
        "Nepodarilo sa načítať žiadny CSV súbor."
    )

# Spojenie aj pri rozdielnych stĺpcoch
df = pd.concat(dataframes, ignore_index=True, sort=False)

print("Rozmer datasetu:", df.shape)
print("Počet zápasov:", len(df))
print("Počet stĺpcov:", len(df.columns))
print("Sezóny:", sorted(df["Season"].unique()))

# Počet zápasov podľa sezóny a ligy
print(
    df.groupby(["Season", "League"])
      .size()
      .unstack(fill_value=0)
)

# Kontrola chýb pri načítaní
print("Počet chýb:", len(errors))
if errors:
    print(pd.DataFrame(errors).to_string(index=False))

Rozmer datasetu: (28901, 198)
Počet zápasov: 28901
Počet stĺpcov: 198
Sezóny: ['2010/11', '2011/12', '2012/13', '2013/14', '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25', '2025/26']
League   Bundesliga  La Liga  Ligue 1  Premier League  Serie A
Season                                                        
2010/11         306      380      380             380      380
2011/12         306      380      380             380      380
2012/13         306      380      380             380      383
2013/14         306      380      381             380      380
2014/15         306      380      380             381      381
2015/16         306      380      381             380      381
2016/17         306      380      380             380      380
2017/18         306      380      380             380      380
2018/19         306      380      380             380      380
2019/20         306      380      279             38

In [4]:
# Podiel chýbajúcich hodnôt v každom stĺpci
missing = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .round(1)
)

print(missing.head(30))

Unnamed: 72    100.0
Unnamed: 70    100.0
Unnamed: 71    100.0
CLCA            95.8
CLCD            95.8
CLCH            95.8
CLH             95.6
CLD             95.6
CLA             95.6
LBCA            95.5
LBCD            95.5
LBCH            95.5
1XBH            94.1
1XBA            94.1
1XBD            94.1
BVCA            94.1
BVCH            94.1
BVCD            94.1
BFDCH           94.0
BFDCD           94.0
BFDCA           94.0
BVA             94.0
BVH             94.0
BMGMA           94.0
BMGMD           94.0
BMGMH           94.0
BVD             94.0
BFDA            94.0
BFDD            94.0
BFDH            94.0
dtype: float64


In [8]:
from pathlib import Path
import pandas as pd

file = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches\big5_2526\E0.csv"
)

test = pd.read_csv(file, encoding="utf-8-sig")

print(test.shape)
print(test.columns.tolist())
print(test["FTHG"].isna().mean())
print(test["FTHG"].head(20))

(380, 132)
['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BFDH', 'BFDD', 'BFDA', 'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BFEH', 'BFED', 'BFEA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BFE>2.5', 'BFE<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'BFEAHH', 'BFEAHA', 'B365CH', 'B365CD', 'B365CA', 'BFDCH', 'BFDCD', 'BFDCA', 'BMGMCH', 'BMGMCD', 'BMGMCA', 'BVCH', 'BVCD', 'BVCA', 'BWCH', 'BWCD', 'BWCA', 'CLCH', 'CLCD', 'CLCA', 'LBCH', 'LBCD', 'LBCA', 'PSCH', 'PSCD', 'PSCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'BFECH', 'BFECD', 'BFECA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.

In [9]:
print("Počet riadkov:", len(df))
print("Počet stĺpcov:", len(df.columns))

# Duplicitné názvy stĺpcov
duplicates = df.columns[df.columns.duplicated()].tolist()

print("\nDuplicitné stĺpce:")
print(duplicates)

Počet riadkov: 28901
Počet stĺpcov: 198

Duplicitné stĺpce:
[]


In [15]:
missing_summary = pd.DataFrame({
    "missing": df.isna().sum(),
    "total": len(df),
})

missing_summary["missing_pct"] = (
    missing_summary["missing"] / missing_summary["total"] * 100
).round(2)

missing_summary["available_pct"] = (
    100 - missing_summary["missing_pct"]
).round(2)

missing_summary = missing_summary.sort_values(
    "missing_pct",
    ascending=False
)

print(missing_summary.head(30))

             missing  total  missing_pct  available_pct
Unnamed: 72    28901  28901       100.00           0.00
Unnamed: 70    28901  28901       100.00           0.00
Unnamed: 71    28901  28901       100.00           0.00
CLCA           27673  28901        95.75           4.25
CLCD           27673  28901        95.75           4.25
CLCH           27673  28901        95.75           4.25
CLH            27621  28901        95.57           4.43
CLD            27621  28901        95.57           4.43
CLA            27621  28901        95.57           4.43
LBCA           27591  28901        95.47           4.53
LBCD           27591  28901        95.47           4.53
LBCH           27591  28901        95.47           4.53
1XBH           27196  28901        94.10           5.90
1XBA           27196  28901        94.10           5.90
1XBD           27196  28901        94.10           5.90
BVCA           27188  28901        94.07           5.93
BVCH           27188  28901        94.07        

In [16]:
df = df.dropna(axis=1, how="all")

In [17]:
print(df.shape)

(28901, 195)


In [18]:
stats = [
    "FTHG", "FTAG",
    "HS", "AS",
    "HST", "AST",
    "HF", "AF",
    "HC", "AC",
    "HY", "AY",
    "HR", "AR",
    "B365H", "B365D", "B365A"
]

availability = (
    df.groupby("Season")[stats]
      .apply(lambda x: x.notna().mean() * 100)
      .round(1)
)

print(availability)

          FTHG   FTAG     HS     AS    HST    AST     HF     AF     HC     AC  \
Season                                                                          
2010/11  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0   
2011/12  100.0  100.0  100.0  100.0  100.0  100.0   99.9   99.9  100.0  100.0   
2012/13   99.8   99.8   99.8   99.8   99.8   99.8   99.8   99.8   99.8   99.8   
2013/14   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   
2014/15   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   
2015/16   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   
2016/17  100.0  100.0   99.9   99.9   99.9   99.9   99.9   99.9   99.9   99.9   
2017/18  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0   
2018/19  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0   
2019/20  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0   
2020/21  100.0  100.0  100.0